# Modelagem preditiva NPS

Este notebook tem como objetivo demonstrar como **modelos de regressão** podem auxiliar na predição de notas NPS, bem como mostrar como **modelos de classificação** conseguem rotular quais são os tipos de clientes que ficarão satisfeitos ou insatisfeitos com o e-commerce.

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


warnings.filterwarnings("ignore")

In [73]:
df = pd.read_csv('../data/raw/desafio_nps_fase_1.csv')

In [74]:
df.columns

Index(['customer_id', 'customer_age', 'customer_region',
       'customer_tenure_months', 'order_id', 'order_value', 'items_quantity',
       'discount_value', 'payment_installments', 'delivery_time_days',
       'delivery_delay_days', 'freight_value', 'delivery_attempts',
       'customer_service_contacts', 'resolution_time_days', 'nps_score',
       'repeat_purchase_30d', 'complaints_count', 'csat_internal_score'],
      dtype='str')

In [75]:
features = [
    'delivery_delay_days', 
    'complaints_count', 
    'customer_service_contacts', 
    'csat_internal_score',
    'repeat_purchase_30d'
]

X = df[features]
y = df['nps_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Teste com modelos de regressão

In [76]:
modelos_reg = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100, max_depth=8),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

resultados_reg = []

print("Resultados dos Testes de Regressão:\n" + "="*40)

for nome, modelo in modelos_reg.items():
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    resultados_reg.append({
        'Modelo': nome, 
        'R² Score': r2, 
        'MSE': mse, 
        'RMSE': rmse
    })
    
    print(f"--- Avaliação: {nome} ---")
    print(f"R² Score: {r2:.2f}")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}\n")

df_comparativo_reg = pd.DataFrame(resultados_reg).sort_values(by='R² Score', ascending=False)

print("="*45)
print("RANKING FINAL DE REGRESSORES:")
print(df_comparativo_reg.to_string(index=False))

Resultados dos Testes de Regressão:
--- Avaliação: Regressão Linear ---
R² Score: 0.64
MSE: 2.25
RMSE: 1.50

--- Avaliação: Random Forest ---
R² Score: 0.63
MSE: 2.32
RMSE: 1.52

--- Avaliação: Gradient Boosting ---
R² Score: 0.64
MSE: 2.26
RMSE: 1.50

RANKING FINAL DE REGRESSORES:
           Modelo  R² Score      MSE     RMSE
 Regressão Linear  0.644567 2.245881 1.498626
Gradient Boosting  0.641981 2.262220 1.504068
    Random Forest  0.633607 2.315132 1.521556


## Teste com classificadores

In [77]:
df['target'] = df['nps_score'].apply(lambda x: 1 if x >= 9 else 0)

In [78]:
X_classificacao = df[features]
y_classificacao = df['target'] 

In [79]:
X_train_classificacao, X_test_classificacao, y_train_classificacao, y_test_classificacao = train_test_split(X_classificacao, y_classificacao, test_size=0.2, random_state=42)

In [80]:
modelos = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (RBF Kernel)": SVC(probability=True, random_state=42)
}

print("Resultados dos Testes de Classificação:\n" + "="*40)

resultados_resumo = []

for nome, modelo in modelos.items():
    modelo.fit(X_train_classificacao, y_train_classificacao)
    y_pred_classificacao = modelo.predict(X_test_classificacao)
    
    acc = accuracy_score(y_test_classificacao, y_pred_classificacao)
    report = classification_report(y_test_classificacao, y_pred_classificacao, output_dict=True)
    
    recall_0 = report['0']['recall']
    
    resultados_resumo.append({'Modelo': nome, 'Acurácia': acc, 'Recall Insatisfeito': recall_0})
    
    print(f"\n> MODELO: {nome}")
    print(f"Acurácia: {acc:.2f}")
    print(f"Matriz de Confusão:\n{confusion_matrix(y_test_classificacao, y_pred_classificacao)}")
    print(classification_report(y_test_classificacao, y_pred_classificacao))

df_comparativo = pd.DataFrame(resultados_resumo).sort_values(by='Acurácia', ascending=False)
print("\n" + "="*40)
print("RANKING FINAL DE MODELOS:")
print(df_comparativo)

Resultados dos Testes de Classificação:

> MODELO: Decision Tree
Acurácia: 0.97
Matriz de Confusão:
[[473   7]
 [  7  13]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       480
           1       0.65      0.65      0.65        20

    accuracy                           0.97       500
   macro avg       0.82      0.82      0.82       500
weighted avg       0.97      0.97      0.97       500


> MODELO: Random Forest
Acurácia: 0.96
Matriz de Confusão:
[[472   8]
 [ 13   7]]
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       480
           1       0.47      0.35      0.40        20

    accuracy                           0.96       500
   macro avg       0.72      0.67      0.69       500
weighted avg       0.95      0.96      0.96       500


> MODELO: Gradient Boosting
Acurácia: 0.95
Matriz de Confusão:
[[470  10]
 [ 13   7]]
              precision    recall  f1-score   support